In [1]:
import subprocess, sys
# 确认包是否已安装
pkgs = ["yfinance", "pandas", "numpy", "matplotlib"]
for pkg in pkgs:
    result = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                            capture_output=True, text=True)
    status = "✅" if result.returncode == 0 else "❌ 缺失"
    print(f"{status}  {pkg}")

❌ 缺失  yfinance
✅  pandas
✅  numpy
✅  matplotlib


In [2]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "yfinance"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
print(result.stderr[-300:] if result.stderr else "")

0
Successfully built multitasking

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [curl_cffi]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [yfinance]





In [3]:
import yfinance as yf
import pandas as pd

# 拉取 5 只股票，最近 3 个月日线数据
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
raw = yf.download(tickers, period="3mo", interval="1d", auto_adjust=True, progress=False)

print("shape:", raw.shape)
print("columns (前10):", raw.columns.tolist()[:10])
print("date range:", raw.index[0].date(), "→", raw.index[-1].date())
print("\n收盘价后5行：")
print(raw["Close"].tail())

shape: (61, 25)
columns (前10): [('Close', 'AAPL'), ('Close', 'AMZN'), ('Close', 'GOOGL'), ('Close', 'MSFT'), ('Close', 'NVDA'), ('High', 'AAPL'), ('High', 'AMZN'), ('High', 'GOOGL'), ('High', 'MSFT'), ('High', 'NVDA')]
date range: 2026-01-14 → 2026-04-13

收盘价后5行：
Ticker            AAPL        AMZN       GOOGL        MSFT        NVDA
Date                                                                  
2026-04-07  253.500000  213.770004  305.459991  372.290009  178.100006
2026-04-08  258.899994  221.250000  317.320007  374.329987  182.080002
2026-04-09  260.489990  233.649994  318.489990  373.070007  183.910004
2026-04-10  260.480011  238.380005  317.239990  370.869995  188.630005
2026-04-13  259.200012  239.889999  321.309998  384.369995  189.309998


In [4]:
import numpy as np

# 取收盘价，整理成干净的 DataFrame
close = raw["Close"].copy()
volume = raw["Volume"].copy()

def make_features(close, volume):
    feats = pd.DataFrame(index=close.index)
    
    # === 动量类 ===
    feats["mom_5d"]  = close.pct_change(5)
    feats["mom_10d"] = close.pct_change(10)
    feats["mom_20d"] = close.pct_change(20)
    
    # === 波动率类 ===
    feats["vol_10d"] = close.pct_change().rolling(10).std()
    feats["vol_20d"] = close.pct_change().rolling(20).std()
    
    # === 相对强弱 ===
    feats["rsi_dist"] = (close - close.rolling(20).mean()) / close.rolling(20).std()
    feats["high_20d_ratio"] = close / close.rolling(20).max()  # 现价/20日最高点
    
    # === 成交量类 ===
    feats["vol_ratio"] = volume / volume.rolling(10).mean()
    
    return feats

aapl_feat = make_features(close["AAPL"], volume["AAPL"])
print("特征 shape:", aapl_feat.shape)
print("\n最后5行：")
print(aapl_feat.tail())
print("\nNaN 数量：")
print(aapl_feat.isna().sum())

特征 shape: (61, 8)

最后5行：
              mom_5d   mom_10d   mom_20d   vol_10d   vol_20d  rsi_dist  \
Date                                                                     
2026-04-07  0.027855  0.007992 -0.024550  0.014157  0.013222  0.084739   
2026-04-08  0.020135  0.028851 -0.007399  0.015555  0.014118  1.527849   
2026-04-09  0.019012  0.031153 -0.001227  0.015586  0.014191  1.965678   
2026-04-10  0.017818  0.030013  0.018455  0.015607  0.013439  1.755896   
2026-04-13  0.001314  0.041801  0.036303  0.014426  0.012398  1.291203   

            high_20d_ratio  vol_ratio  
Date                                   
2026-04-07        0.971897   1.496833  
2026-04-08        0.992677   0.998177  
2026-04-09        1.000000   0.684687  
2026-04-10        0.999962   0.781126  
2026-04-13        0.995048   0.905615  

NaN 数量：
mom_5d             5
mom_10d           10
mom_20d           20
vol_10d           10
vol_20d           20
rsi_dist          19
high_20d_ratio    19
vol_ratio          9

In [5]:
# 对所有股票批量生成特征，拼成一个长表
all_feats = []

for ticker in tickers:
    feat = make_features(close[ticker], volume[ticker])
    feat["ticker"] = ticker
    feat = feat.reset_index()  # 把 Date 从 index 变成列
    all_feats.append(feat)

df = pd.concat(all_feats, ignore_index=True)
df = df.dropna()

print("总行数（dropna后）:", len(df))
print("每只股票行数：")
print(df["ticker"].value_counts().sort_index())
print("\n特征列：", [c for c in df.columns if c not in ["Date", "ticker"]])
print("\n数值范围：")
print(df.drop(columns=["Date", "ticker"]).describe().round(4))

总行数（dropna后）: 205
每只股票行数：
ticker
AAPL     41
AMZN     41
GOOGL    41
MSFT     41
NVDA     41
Name: count, dtype: int64

特征列： ['mom_5d', 'mom_10d', 'mom_20d', 'vol_10d', 'vol_20d', 'rsi_dist', 'high_20d_ratio', 'vol_ratio']

数值范围：
         mom_5d   mom_10d   mom_20d   vol_10d   vol_20d  rsi_dist  \
count  205.0000  205.0000  205.0000  205.0000  205.0000  205.0000   
mean    -0.0010   -0.0131   -0.0359    0.0183    0.0190   -0.3989   
std      0.0413    0.0549    0.0604    0.0057    0.0049    1.1447   
min     -0.1037   -0.1743   -0.1730    0.0085    0.0102   -2.8616   
25%     -0.0285   -0.0420   -0.0788    0.0140    0.0147   -1.0930   
50%     -0.0037   -0.0127   -0.0357    0.0173    0.0193   -0.5054   
75%      0.0205    0.0148    0.0051    0.0225    0.0221    0.2100   
max      0.1364    0.2034    0.1552    0.0342    0.0302    3.2030   

       high_20d_ratio  vol_ratio  
count        205.0000   205.0000  
mean           0.9346     0.9497  
std            0.0473     0.2500  
min     

In [6]:
# 计算每只股票的次日收益率（作为排序 label）
next_day_ret = close.pct_change().shift(-1)  # shift(-1) = 用明天的收益率标注今天

# 拼入 df
ret_long = next_day_ret.stack().reset_index()
ret_long.columns = ["Date", "ticker", "next_ret"]

df = df.merge(ret_long, on=["Date", "ticker"], how="left")
df = df.dropna(subset=["next_ret"])

print("加入 next_ret 后行数:", len(df))
print("\n每只股票行数：")
print(df["ticker"].value_counts().sort_index())
print("\nnext_ret 统计：")
print(df["next_ret"].describe().round(4))
print("\n某一天5只股票的排序示例（取第一个交易日）：")
sample_date = df["Date"].iloc[0]
print(df[df["Date"] == sample_date][["ticker", "next_ret"]].sort_values("next_ret", ascending=False))

加入 next_ret 后行数: 200

每只股票行数：
ticker
AAPL     40
AMZN     40
GOOGL    40
MSFT     40
NVDA     40
Name: count, dtype: int64

next_ret 统计：
count    200.0000
mean       0.0011
std        0.0182
min       -0.0546
25%       -0.0097
50%        0.0011
75%        0.0115
max        0.0560
Name: next_ret, dtype: float64

某一天5只股票的排序示例（取第一个交易日）：
    ticker  next_ret
41    MSFT -0.001294
123   AMZN -0.004058
82   GOOGL -0.010615
164   NVDA -0.022093
0     AAPL -0.022733


In [7]:
# 每个交易日内，对5只股票按 next_ret 排名（rank 1 = 当天最强）
df["rank"] = df.groupby("Date")["next_ret"].rank(ascending=False).astype(int)

print("rank 分布（应该每个值都出现40次）：")
print(df["rank"].value_counts().sort_index())

print("\n同一天的完整示例：")
sample_date = df["Date"].iloc[0]
cols = ["ticker", "next_ret", "rank"]
print(df[df["Date"] == sample_date][cols].sort_values("rank"))

rank 分布（应该每个值都出现40次）：
rank
1    40
2    40
3    40
4    40
5    40
Name: count, dtype: int64

同一天的完整示例：
    ticker  next_ret  rank
41    MSFT -0.001294     1
123   AMZN -0.004058     2
82   GOOGL -0.010615     3
164   NVDA -0.022093     4
0     AAPL -0.022733     5


In [8]:
import os

save_dir = os.path.expanduser("~/ml-projects/stock-ranker/data")
os.makedirs(save_dir, exist_ok=True)

df.to_csv(f"{save_dir}/features_labeled.csv", index=False)

print("已保存：", f"{save_dir}/features_labeled.csv")
print("shape:", df.shape)
print("列名:", df.columns.tolist())

已保存： /Users/tongwenchao/ml-projects/stock-ranker/data/features_labeled.csv
shape: (200, 12)
列名: ['Date', 'mom_5d', 'mom_10d', 'mom_20d', 'vol_10d', 'vol_20d', 'rsi_dist', 'high_20d_ratio', 'vol_ratio', 'ticker', 'next_ret', 'rank']
